# Day 2 — Hopcroft–Karp 复杂二分图：多分支增广路推演

> **日期**：2026-07-04
> **主题**：从简单链到多分支竞争——HK 算法在非平凡二分图上的完整行为

## 一、重新搭建更复杂二分图

> 相比 Day 1 的简单案例，本次图的核心变化：**每个工人多岗位选择 + 存在多条独立增广分支**，消除"单一通路"的理想化假设。

### 1.1 顶点与邻接关系

| 集合 | 顶点 | 含义 |
|:---:|:---:|:---|
| 左 $X$ | $x_1,x_2,x_3,x_4,x_5$ | 工人（Worker） |
| 右 $Y$ | $y_1,y_2,y_3,y_4,y_5$ | 岗位（Job） |

每个工人至少提供 **2 个及以上**可选岗位，确保存在多条潜在增广分支：

$$
\begin{aligned}
x_1 &: y_1,\ y_2,\ y_3 \quad \text{(3 个岗位，选择丰富)} \\
x_2 &: y_1 \\
x_3 &: y_2,\ y_4 \\
x_4 &: y_3,\ y_4 \\
x_5 &: y_5
\end{aligned}
$$

### 1.2 数组约定（沿用 Day 1 规则）

| 数组 | 含义 |
|:---:|:---|
| $mx[u]$ | 工人 $x_u$ 当前匹配的岗位编号，$0$ = 自由 |
| $my[v]$ | 岗位 $y_v$ 当前匹配的工人编号，$0$ = 自由 |
| $dx[u]$ | BFS 分层距离，$\infty$ = 本轮不可达 |
| $ans$ | 当前匹配总数 |

> 下标 $0$ 仅作占位，无实际含义。

### 1.3 初始匹配状态

人为构造局部匹配，刻意制造**多条置换分支**：

$$
\text{初始配对：}\quad x_1 \leftrightarrow y_1,\quad x_3 \leftrightarrow y_2,\quad x_4 \leftrightarrow y_3
$$

| 状态 | 成员 |
|:---|:---|
| 自由工人 | $x_2,\ x_5$ |
| 自由岗位 | $y_4,\ y_5$ |

对应数组状态：

$$
\begin{aligned}
mx &= [0,\ \mathbf{1},\ 0,\ \mathbf{2},\ \mathbf{3},\ 0] \\
my &= [0,\ \mathbf{1},\ \mathbf{3},\ \mathbf{4},\ 0,\ 0] \\
ans &= 3
\end{aligned}
$$

### 1.4 当前局面难点

自由工人 $x_2$ **只能**走 $y_1$，直接挤占 $x_1$。但 $x_1$ 手握**两条**备选置换路线：

```
路线 A: x₁ → y₂ → 挤占 x₃ → x₃ 可换 y₄（空岗）
路线 B: x₁ → y₃ → 挤占 x₄ → x₄ 可换 y₄（空岗）
```

> 两条完全**独立**的深层置换分支，DFS 按邻点顺序遍历、**自动筛选**合法最短路径，不会同时走两条。

## 二、第 1 阶段：BFS 分层

> BFS 的职责：**不做选择，只划定所有满足最短层级的节点**，多条分支同时保留，交给 DFS 裁决。

### Step 1 — 初始化

遍历 $u = 1 \sim 5$，找出自由工人：

$$
mx[2] = 0,\quad mx[5] = 0
$$

- $dx[2] = 0,\ dx[5] = 0$，二者入队
- $dx[1] = dx[3] = dx[4] = \infty$
- `has_aug = false`
- 初始队列：$[x_2,\ x_5]$

### Step 2 — 逐层广度扩散

#### 🔹 第一层：取出 $x_2$（$dx = 0$）

| 邻点 | $my[v]$ | 操作 |
|:---|:---|:---|
| $y_1$ | $x_1$（饱和） | $dx[x_1] = \infty$ → 更新 $dx[1] = 1$，$x_1$ 入队 |

#### 🔹 第一层：取出 $x_5$（$dx = 0$）

| 邻点 | $my[v]$ | 操作 |
|:---|:---|:---|
| $y_5$ | $0$（空岗） | `has_aug = true`，无需继续分层 |

队列现为：$[x_1]$

#### 🔹 第二层：取出 $x_1$（$dx = 1$）

邻点依次 $y_1,\ y_2,\ y_3$：

| 邻点 | $my[v]$ | 操作 |
|:---|:---|:---|
| $y_1$ | — | 自身匹配边，跳过 |
| $y_2$ | $x_3$ | $dx[3] = \infty$ → $dx[3] = 2$，$x_3$ 入队 |
| $y_3$ | $x_4$ | $dx[4] = \infty$ → $dx[4] = 2$，$x_4$ 入队 |

队列现为：$[x_3,\ x_4]$

#### 🔹 第三层：取出 $x_3$（$dx = 2$）

| 邻点 | $my[v]$ | 操作 |
|:---|:---|:---|
| $y_2$ | — | 自身匹配边，跳过 |
| $y_4$ | $0$（空岗） | `has_aug = true` |

#### 🔹 第三层：取出 $x_4$（$dx = 2$）

| 邻点 | $my[v]$ | 操作 |
|:---|:---|:---|
| $y_3$ | — | 自身匹配边，跳过 |
| $y_4$ | $0$（空岗） | `has_aug = true` |

### Step 3 — 分层结果

$$
dx = [\infty,\ \mathbf{1},\ \mathbf{0},\ \mathbf{2},\ \mathbf{2},\ \mathbf{0}]
$$

**层级含义**：

| $dx$ 值 | 层 | 顶点 | 语义 |
|:---:|:---:|:---|:---|
| $0$ | 起点层 | $x_2,\ x_5$ | 两条独立增广路起点 |
| $1$ | 第 1 层 | $x_1$ | $x_2$ 挤占的直接后继 |
| $2$ | 第 2 层 | $x_3,\ x_4$ | $x_1$ 的下一层可选置换目标 |

**本轮三条最短增广路**：

| 通路 | 路径 | 类型 |
|:---|:---|:---|
| 通路 1 | $x_2 \to y_1 \to x_1 \to y_2 \to x_3 \to y_4$ | $x_1$ 走 $y_2$ 分支 |
| 通路 2 | $x_2 \to y_1 \to x_1 \to y_3 \to x_4 \to y_4$ | $x_1$ 走 $y_3$ 分支 |
| 通路 3 | $x_5 \to y_5$ | 独立直达空岗 |

> **关键认知**：BFS 不会主动选择分支。它把**所有满足最短层级**的节点全部记入 $dx$，多条分支同时保留，DFS 阶段再用邻点顺序裁决。

## 三、第 2 阶段：批量 DFS

> DFS 通行规则回顾：对工人 $u$、邻点 $v$，令 $u' = my[v]$，仅当 $u' = 0$ **或** $dx[u'] = dx[u] + 1$ 时允许进入。

### 3.1 $DFS(x_2)$ — 存在 $y_2$ / $y_3$ 两条置换分支

#### 🔹 $DFS(x_2),\ dx = 0$

邻点仅 $y_1$：

$$
u' = my[y_1] = x_1,\quad dx[x_1] = 1 = 0 + 1 \quad \checkmark
$$

满足分层条件 → 递归 $DFS(x_1)$

#### 🔹 $DFS(x_1),\ dx = 1$

邻点遍历顺序：$y_1 \to y_2 \to y_3$

| 邻点 | 判断 | 结果 |
|:---|:---|:---|
| $y_1$ | 当前匹配边 | ⏭ 跳过 |
| $y_2$ | $u' = x_3,\ dx[x_3] = 2 = 1 + 1$ ✓ | → 递归 $DFS(x_3)$ |
| $y_3$ | — | ⛔ 不再遍历 |

#### 🔹 $DFS(x_3),\ dx = 2$

| 邻点 | 判断 | 结果 |
|:---|:---|:---|
| $y_2$ | 匹配边 | ⏭ 跳过 |
| $y_4$ | $my[y_4] = 0$，空岗！ | 🎯 找到增广路 |

#### 🔹 回溯逐层更新匹配

```
DFS(x₃) → true:  mx[3] = 4,  my[4] = 3
DFS(x₁) → true:  mx[1] = 2,  my[2] = 1
DFS(x₂) → true:  mx[2] = 1,  my[1] = 2
```

### 3.2 $DFS(x_5)$ — 独立无冲突直达

#### 🔹 $DFS(x_5),\ dx = 0$

邻点 $y_5$，$my[y_5] = 0$（空岗）→ 直接匹配：

$$
mx[5] = 5,\quad my[5] = 5 \quad \rightarrow \text{true}
$$

### 3.3 DFS 阶段结束

全局匹配数组：

$$
\begin{aligned}
mx &= [0,\ \mathbf{2},\ \mathbf{1},\ \mathbf{4},\ \mathbf{3},\ \mathbf{5}] \\
my &= [0,\ \mathbf{2},\ \mathbf{1},\ \mathbf{4},\ \mathbf{3},\ \mathbf{5}] \\
ans &= 3 + 2 = 5
\end{aligned}
$$

**最终配对**：

| 工人 | 岗位 | 来源 |
|:---:|:---:|:---|
| $x_1$ | $y_2$ | 被 $x_2$ 挤占后走 $y_2$ 分支重配 |
| $x_2$ | $y_1$ | 本轮新增（挤占 $x_1$） |
| $x_3$ | $y_4$ | 被 $x_1$ 挤占后去空岗 |
| $x_4$ | $y_3$ | 未受影响 |
| $x_5$ | $y_5$ | 本轮新增（直达空岗） |

所有工人、岗位全部饱和 ✅

## 四、分支选择：如果邻接表顺序不同？

> 若 $x_1$ 的邻接表存储顺序调换为 $y_1,\ y_3,\ y_2$，DFS 会优先走 $y_3$ 分支：

```
DFS(x₂) → DFS(x₁)
  x₁ 访问 y₃ → u' = x₄, dx[x₄] = 2 ✓ → DFS(x₄)
    x₄ 访问 y₄（空岗）→ 返回 true
回溯：mx₄ = 4, mx₁ = 3, mx₂ = 1
```

**最终配对变为**：

$$
x_2 \leftrightarrow y_1,\quad x_1 \leftrightarrow y_3,\quad x_4 \leftrightarrow y_4,\quad x_3 \leftrightarrow y_2,\quad x_5 \leftrightarrow y_5
$$

### 核心结论

| 性质 | 说明 |
|:---|:---|
| **两种均为合法最大匹配** | 增广路长度一致，都属于最短路径 |
| **最大匹配不唯一** | 方案可不同，但匹配数 $ans = 5$ 不变 |
| **DFS 只走第一条通路** | 找到一条即回溯，剩余分支自动剪枝 |
| **$dx$ 保证路径等长** | 不存在"一条短一条长"的区别，选哪条都不影响总数 |

## 五、第 2 轮 BFS：终止验证

全部工人 $x_1 \sim x_5$ 的 $mx[u] \neq 0$，无自由左点：

- BFS 队列为空
- `has_aug = false`
- 算法终止

> 🏁 **最终最大匹配数 $ans = 5$**

## 六、核心疑问解答

### Q1：当中间工人 $x_1$ 存在多个可置换岗位时，算法如何选择？

| 机制 | 作用 |
|:---|:---|
| **$dx$ 锁死筛选门槛** | 只有满足 $dx[u'] = dx[u] + 1$ 的岗位才允许遍历，更长路径直接过滤 |
| **邻接表顺序决定优先级** | DFS 按邻点依次判断，找到第一条能抵达空岗的通路立刻回溯 |
| **找到一个即停止** | 单个起点工人只需一条增广路，剩余分支直接剪枝 |
| **多条分支等价** | 同分层下所有分支都是最短路径，无论选哪条，最终匹配总数不变 |

### Q2：为什么不会同时走多条分支造成顶点冲突？

> **HK 定理保证**：同一阶段、同一长度的最短增广路顶点互不相交。

| 场景 | 冲突风险 |
|:---|:---|
| $x_2$ 衍生的置换通路 vs $x_5$ 直达通路 | ✅ 无冲突——完全无共用顶点，可同时批量更新 |
| $x_1$ 下属 $y_2$ / $y_3$ 两条分支 | ✅ 无冲突——共享起点 $x_2$，互斥只能选一条 |

## 七、整套组件配合逻辑（复杂案例复盘）

```
┌──────────────────────────────────────────────────┐
│              HK 算法四组件协作流程                  │
├──────────┬───────────────────────────────────────┤
│ mx / my  │ 提供当前匹配现状                        │
│          │ → BFS 据此识别自由工人与岗位归属          │
├──────────┼───────────────────────────────────────┤
│ BFS + dx │ 统一划定所有合法最短分支                  │
│          │ → 不做选择，只记录层级，消除长路径干扰      │
├──────────┼───────────────────────────────────────┤
│   DFS    │ 按顺序遍历分支，择优匹配                  │
│          │ → 依托 dx 过滤无效绕路                   │
│          │ → 找到第一条可行通路就更新，剪枝多余备选    │
├──────────┼───────────────────────────────────────┤
│ 批量处理 │ 一轮同时处理多条无冲突增广路               │
│          │ → 本轮一次增加 2 组匹配（x₂ 链 + x₅ 直达） │
├──────────┼───────────────────────────────────────┤
│ 迭代终止 │ BFS 检测无增广路 → 算法终止 → 全局最优    │
└──────────┴───────────────────────────────────────┘
```

## 八、与 Day 1 简单例子的核心区别

| 维度 | Day 1 简单案例 | Day 2 复杂案例 |
|:---|:---|:---|
| 中间工人置换 | 仅单一置换岗位 | $x_1$ 提供**两条独立**、等长的置换路线 |
| 分支选择 | 无分支，唯一通路 | 直观展示 DFS **分支取舍规则**与剪枝行为 |
| 独立增广路 | 无 | $x_5 \to y_5$ 体现 HK **批量同步增广**特性 |
| 最大匹配唯一性 | 唯一 | **不唯一**——不同分支选择得到不同配对方案，但总数固定 |

> *推演完成。Day 3 将进入代码实现层面，用 Python 逐行映射上述逻辑。*